In [ ]:
input_data = None
targetpop_data = None
output_data = None
output_model = None
util = None
display_util = None
configfile = "config/config.yml"

In [ ]:
import yaml

with open(configfile) as stream:
    config = yaml.safe_load(stream)

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import matplotlib_inline
import matplotlib.pyplot as plt

from IPython.display import Markdown
import pandera.pandas as pa
from pandera.typing import Series

matplotlib_inline.backend_inline.set_matplotlib_formats("svg")

%matplotlib inline
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", 500)
pd.set_option("display.max_columns", None)
plt.ioff()
plt.rcParams["figure.figsize"] = (8, 5)

sys.path.append(str(Path(util).parent))
sys.path.append(str(Path(display_util).parent))

In [ ]:
from display_util import (  # noqa: E402
    rule_setup,
    display_data_doc,
    display_long_data_doc,
)
from util import (  # noqa: E402
    drop_col_few_distinct,
    TransplantPostOPID,
    drop_duplicate_columns,
    common_translate,
)

### Target Population Filtering

The patients in the dataset were filtered to match the target population (see [](general:tpf)). Afterwards we tried again to remove empty and duplicate columns.

In [ ]:
data = pd.read_parquet(input_data)
rec = data["transplant_et_id"].copy()
targetpop = pd.read_parquet(targetpop_data)
data = data[rec.isin(targetpop["transplant_et_id"])]
display(
    Markdown(
        f"""The filter process reduced the number of transplants in the data ({rec.nunique()}) and target population ({targetpop["transplant_et_id"].nunique()})
            to {rec[rec.isin(targetpop["transplant_et_id"])].nunique()} in the processed data.
        """
    )
)
del targetpop, rec

In [ ]:
data = drop_col_few_distinct(data)
data = drop_duplicate_columns(data)

### Integration of Seperated Institute Data

Only {term}`ET` data is in this file, so no data processing was necessary at this step (see [](general:ic)).

## Domain Steps

For this file the general plan for domain preprocessing of longitudinal data was followed (see [](general:ds)).

### Row Filtering

There is no column differentiating between different types of medication (see [](general:rf)). We kept all rows.

In [ ]:
display_long_data_doc(data, ["transplant_et_id"], "date", None)

### Unit Conversions

Only common translations were applied (see [](general:uc)).

In [ ]:
data = common_translate(data, config["data"]["common_translations"])

### Consolidating Columns

No consolidation was necessary. (see [](general:crc))

## Intermediate Dataset

For this longitudinal dataset we recommend the `date` column as the time axis.

In [ ]:
indcols = ["transplant_et_id"]
data = data.sort_index(axis=1).sort_values(indcols + ["date"], axis=0)
data = data.set_index(indcols)

In [ ]:
class FollowupNiereMedikation(TransplantPostOPID):
    date: Series[float] = pa.Field(
        coerce=True,
        nullable=False,
        unique=False,
        title="Follow-Up Date",
        description="When was the follow-up conducted?",
    )
    immunosuppressant: Series[str] = pa.Field(
        coerce=True,
        nullable=False,
        unique=False,
        title="Immunosuppressant",
        description="Which immunosuppressant was provided?",
    )
    immunosuppressant_details: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Immunosuppressant Detail",
        description="Which exact immunosuppressant was provided?",
    )

    class Config:
        title = "Immunosuppressant Follow-Up Dataset"
        description = "Each row represents a immunosuppressant reported at a follow-up visit. The data is based on the 'element_followup_niere_medikation.csv' file. It contains data from the ET."
        multiindex_strict = True
        multiindex_coerce = True

In [ ]:
display_data_doc(FollowupNiereMedikation, data)

In [ ]:
FollowupNiereMedikation.to_schema().validate(data).to_parquet(output_data)
with open(output_model, "wt") as fh:
    FollowupNiereMedikation.to_yaml(stream=fh)

## Technical Information

In [ ]:
rule_setup(
    {
        "input_data": input_data,
        "targetpop_data": targetpop_data,
        "output_data": output_data,
        "output_model": output_model,
        "util": util,
        "display_util": display_util,
    }
)